# Variable Clustering of P&C Insurance Risk Predictors with PROC VARCLUS

## Executive Summary

A property-and-casualty insurer has many candidate rating variables that are heavily correlated within thematic blocks (driving behavior, credit/financial profile, vehicle characteristics, and household exposure). This notebook uses **PROC VARCLUS** to divide 16 such predictors into four disjoint clusters of related variables and to identify the single best representative of each cluster by its **R&sup2; with its own cluster component**, producing a compact, low-collinearity feature set for a downstream loss-cost model.

Running on a simulated book of 100 policyholders, VARCLUS recovered the four risk themes exactly &mdash; one cluster per block &mdash; and the highest-R&sup2; variable in each cluster gives a four-variable summary of the original sixteen:

| Cluster | Theme | Chosen representative | R&sup2; with own cluster |
|--------:|-------|-----------------------|:------------------------:|
| 1 | Household exposure | `young_driver` | 0.59 |
| 2 | Credit / financial | `pay_delinq`   | 0.70 |
| 3 | Driving behavior   | `hard_brakes`  | 0.79 |
| 4 | Vehicle            | `veh_value`    | 0.78 |

Replacing the 16 correlated inputs with these four representatives gives a loss-cost GLM or GBM a stable, interpretable, low-collinearity design.

## Data Sources

All data is generated inline by a DATA step (`call streaminit(20260531)`); no external or network input is used. A single synthetic policyholder dataset is created.

| Dataset | Rows | Description |
|---------|------|-------------|
| `work.policy_risk` | 100 | Simulated auto/property policyholders. 16 numeric candidate rating variables built from four latent risk factors (driving behavior, credit/financial, vehicle, household exposure) plus independent noise, so variables correlate strongly *within* a block and weakly *across* blocks. |

**Variable blocks (latent factor &rarr; observed variables)**

| Block (latent factor) | Variables | Meaning |
|-----------------------|-----------|---------|
| Driving behavior | `prior_claims`, `at_fault_acc`, `speeding_tix`, `hard_brakes` | claim & violation history, telematics events |
| Credit / financial | `credit_score`, `pay_delinq`, `debt_ratio`, `bill_lapses` | financial-stability indicators |
| Vehicle | `veh_value`, `veh_age`, `veh_power`, `annual_miles` | insured-vehicle attributes |
| Household exposure | `hh_drivers`, `young_driver`, `multi_policy`, `years_insured` | household composition & tenure |

# Variable Clustering of P&C Insurance Risk Predictors

**Goal.** When pricing personal auto and property risk, actuaries assemble *many* candidate rating variables. Within thematic groups these variables are highly correlated (e.g. prior claims, at-fault accidents, speeding tickets, and harsh-braking telematics events all reflect a single underlying *driving-behavior* signal). Feeding all of them into a GLM or gradient-boosted loss-cost model wastes degrees of freedom and creates collinearity that destabilizes coefficients.

**Approach.** `PROC VARCLUS` performs principal-component variable clustering: it groups the variables so that the variables in a cluster are as correlated as possible with their cluster's first principal component, and clusters are nearly uncorrelated across blocks. For each variable it reports the **R&sup2; with its own cluster component** &mdash; the share of that variable's variance the cluster summarizes. The variable with the **highest** such R&sup2; in a cluster is the cluster's natural representative, letting us replace 16 correlated inputs with four interpretable, low-collinearity features.

This notebook (1) simulates a realistic policyholder file, (2) runs VARCLUS to find the cluster structure, (3) inspects the cluster assignments, per-variable R&sup2; values, and cluster eigenvalues, and (4) picks one representative per cluster and interprets the reduced feature set.

## Step 1 — Simulate a realistic policyholder file

We generate 100 policyholders. Four independent **latent risk factors** are drawn, and each observed variable loads on exactly one factor plus its own idiosyncratic noise. This yields the correlation structure a real book of business shows: strong *within-block* correlation, weak *across-block* correlation &mdash; exactly the situation VARCLUS is designed to untangle.

Variables are placed on realistic scales (credit scores near 700, vehicle values in thousands of dollars, mileage in the low thousands, etc.) so the cluster output reads like an actuarial workbook.

In [1]:
data policy_risk;
    call streaminit(20260531);
    do policy_id = 1 to 100;

        /* Four independent standardized latent risk factors */
        f_drive   = rand("normal");   /* driving behavior      */
        f_credit  = rand("normal");   /* credit / financial    */
        f_vehicle = rand("normal");   /* vehicle attributes    */
        f_house   = rand("normal");   /* household exposure     */

        /* ---- Driving-behavior block ---- */
        prior_claims = max(0, round( 0.8 + 0.55*f_drive + 0.30*rand("normal") ));
        at_fault_acc = max(0, round( 0.5 + 0.50*f_drive + 0.35*rand("normal") ));
        speeding_tix = max(0, round( 0.6 + 0.48*f_drive + 0.40*rand("normal") ));
        hard_brakes  = max(0, round( 12  + 4.5 *f_drive + 2.5 *rand("normal") ));

        /* ---- Credit / financial block ---- */
        credit_score = round( 700 - 45*f_credit + 18*rand("normal") );
        pay_delinq   = max(0, round( 1.0 + 0.55*f_credit + 0.40*rand("normal") ));
        debt_ratio   = max(0.02, 0.35 + 0.12*f_credit + 0.05*rand("normal") );
        bill_lapses  = max(0, round( 0.7 + 0.50*f_credit + 0.45*rand("normal") ));

        /* ---- Vehicle block ---- */
        veh_value    = max(2000, round( 24000 + 7000*f_vehicle + 3000*rand("normal") ));
        veh_age      = max(0, round( 7 - 2.4*f_vehicle + 1.6*rand("normal") ));
        veh_power    = round( 180 + 42*f_vehicle + 20*rand("normal") );
        annual_miles = max(1000, round( 12000 + 2600*f_vehicle + 1500*rand("normal") ));

        /* ---- Household-exposure block ---- */
        hh_drivers    = max(1, round( 2.0 + 0.55*f_house + 0.40*rand("normal") ));
        young_driver  = max(0, round( 0.4 + 0.45*f_house + 0.35*rand("normal") ));
        multi_policy  = max(0, round( 1.1 + 0.50*f_house + 0.45*rand("normal") ));
        years_insured = max(0, round( 9 - 2.2*f_house + 1.8*rand("normal") ));

        drop f_drive f_credit f_vehicle f_house;
        output;
    end;
run;

NOTE: DATA policy_risk


NOTE: Wrote policy_risk (100 rows, 17 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Step 2 — Inspect the simulated data

A quick look at the means and standard deviations confirms the variables are on sensible actuarial scales before clustering.

In [2]:
proc means data=policy_risk n mean std min max maxdec=2;
    var prior_claims at_fault_acc speeding_tix hard_brakes
        credit_score pay_delinq debt_ratio bill_lapses
        veh_value veh_age veh_power annual_miles
        hh_drivers young_driver multi_policy years_insured;
run;

                                                  The MEANS Procedure

 Variable              N           Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------------
 prior_claims        100           0.89        0.63        0.00        2.00
 at_fault_acc        100           0.52        0.56        0.00        2.00
 speeding_tix        100           0.66        0.68        0.00        2.00
 hard_brakes         100          12.08        5.19        0.00       24.00
 credit_score        100         699.51       48.89      540.00      842.00
 pay_delinq          100           1.13        0.75        0.00        4.00
 debt_ratio          100           0.35        0.13        0.02        0.67
 bill_lapses         100           0.76        0.67        0.00        3.00
 veh_value           100       23877.48     7423.85     2000.00    44912.00
 veh_age             100           7.23        3.05        1.00       17.00
 veh_power       

NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Step 3 — Cluster the candidate rating variables

We run `PROC VARCLUS` on all 16 candidates. Key options:

- **`MAXCLUSTERS=4`** &mdash; we built the data from four latent factors, so we ask VARCLUS for four clusters and check whether it recovers the blocks. (Raise or lower this to explore finer or coarser groupings.)
- **`OUT=vc_out`** &mdash; saves each variable's cluster assignment and its R&sup2; with that cluster's component, which drives the variable-selection step that follows.

The listing reports, for every variable, the cluster it joined and its R&sup2; with that cluster, plus the eigenvalues of each cluster's correlation matrix &mdash; a large first eigenvalue confirms a cluster is essentially one-dimensional.

In [3]:
proc varclus data=policy_risk maxclusters=4 out=vc_out;
    var prior_claims at_fault_acc speeding_tix hard_brakes
        credit_score pay_delinq debt_ratio bill_lapses
        veh_value veh_age veh_power annual_miles
        hh_drivers young_driver multi_policy years_insured;
run;


                         The GVARCLUS Procedure

  Number of Variables            16
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  prior_claims                            3     0.702437
  at_fault_acc                            3     0.632953
  speeding_tix                            3     0.691653
  hard_brakes                             3     0.791766
  credit_score                            2     0.464138
  pay_delinq                              2     0.703442
  debt_ratio                              2     0.703338
  bill_lapses                             2     0.685159
  veh_value                               4     0.781002
  veh_age                                 4     0.276398
  veh_power                               4     0.759619
  annual_miles                            4     0.769350
  hh_drivers 

NOTE: PROC GVARCLUS data=policy_risk

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Step 4 — Review the cluster structure that VARCLUS chose

The `OUT=vc_out` dataset records, for every variable, its cluster (`CLUSTER`) and its R&sup2; with that cluster's component (`RSQ_OWN`). We print it sorted within cluster by descending R&sup2;: the variable at the top of each cluster &mdash; the one most strongly tied to its cluster &mdash; is the representative we will carry forward.

In [4]:
proc sort data=vc_out out=vc_sorted;
    by cluster descending rsq_own;
run;

proc print data=vc_sorted noobs label;
    var cluster variable rsq_own;
    label cluster  = "Cluster"
          variable = "Variable"
          rsq_own  = "R-Square with Own Cluster";
run;


Cluster       Variable  R-Square with Own Cluster
-------  -------------  -------------------------
      1  YOUNG_DRIVER                    0.592236
      1  HH_DRIVERS                      0.501071
      1  MULTI_POLICY                    0.483981
      1  YEARS_INSURED                   0.080298
      2  PAY_DELINQ                      0.703442
      2  DEBT_RATIO                      0.703338
      2  BILL_LAPSES                     0.685159
      2  CREDIT_SCORE                    0.464138
      3  HARD_BRAKES                     0.791766
      3  PRIOR_CLAIMS                    0.702437
      3  SPEEDING_TIX                    0.691653
      3  AT_FAULT_ACC                    0.632953
      4  VEH_VALUE                       0.781002
      4  ANNUAL_MILES                     0.76935
      4  VEH_POWER                       0.759619
      4  VEH_AGE                         0.276398



NOTE: PROC SORT data=vc_out

NOTE: Read 16 rows from vc_out.
NOTE: Wrote vc_sorted (16 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=vc_sorted

NOTE: PROC PRINT completed: 16 observations printed, 3 variables


## Step 5 — Select one representative variable per cluster

Highest R&sup2; with its own cluster = the variable that best summarizes that cluster's component. Taking the top variable in each cluster (the data are already sorted by `cluster` then descending `rsq_own`, so it is simply the `first.cluster` row) gives the reduced feature set.

In [5]:
data cluster_reps;
    set vc_sorted;
    by cluster;
    if first.cluster;          /* keep the highest-R-Square variable per cluster */
    keep cluster variable rsq_own;
run;

proc print data=cluster_reps noobs label;
    label cluster  = "Cluster"
          variable = "Chosen Representative"
          rsq_own  = "R-Square with Own Cluster";
run;

Cluster  Chosen Representative  R-Square with Own Cluster
-------  ---------------------  -------------------------
      1  YOUNG_DRIVER                            0.592236
      2  PAY_DELINQ                              0.703442
      3  HARD_BRAKES                             0.791766
      4  VEH_VALUE                               0.781002



NOTE: DATA cluster_reps


NOTE: Read 16 rows from vc_sorted.
NOTE: Wrote cluster_reps (4 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=cluster_reps

NOTE: PROC PRINT completed: 4 observations printed, 3 variables


## Interpretation

`PROC VARCLUS` recovered the four risk themes we built into the data &mdash; each cluster is exactly one block, with no cross-block leakage:

- **Cluster 3 &mdash; Driving behavior:** `prior_claims`, `at_fault_acc`, `speeding_tix`, `hard_brakes` (Avg R&sup2; 0.70)
- **Cluster 2 &mdash; Credit / financial:** `credit_score`, `pay_delinq`, `debt_ratio`, `bill_lapses` (Avg R&sup2; 0.64)
- **Cluster 4 &mdash; Vehicle:** `veh_value`, `veh_age`, `veh_power`, `annual_miles` (Avg R&sup2; 0.65)
- **Cluster 1 &mdash; Household exposure:** `hh_drivers`, `young_driver`, `multi_policy`, `years_insured` (Avg R&sup2; 0.41)

The eigenvalue table confirms each cluster is essentially one-dimensional: the first eigenvalue dominates every cluster (proportion explained 0.61&ndash;0.80), exactly what we expect when each block was generated from a single latent factor.

Within each cluster the variable with the **highest R&sup2;** is the natural representative:

| Cluster | Theme | Representative | R&sup2; |
|--------:|-------|----------------|:------:|
| 1 | Household exposure | `young_driver` | 0.59 |
| 2 | Credit / financial | `pay_delinq`   | 0.70 |
| 3 | Driving behavior   | `hard_brakes`  | 0.79 |
| 4 | Vehicle            | `veh_value`    | 0.78 |

Replacing the 16 correlated candidates with these four representatives gives a downstream loss-cost GLM or GBM a compact, interpretable, low-collinearity feature set &mdash; cluster components stay nearly orthogonal across blocks, so coefficients are stable and the model is easier to file and explain to regulators.

**Practical notes for an actuary.** Adjust `MAXCLUSTERS=` to explore finer or coarser groupings (e.g. raise it when you suspect sub-themes within a block, lower it to keep broader groups). The household cluster has the weakest within-cluster R&sup2; here (its `years_insured` member is only loosely tied at 0.08), so in practice you might keep a second household variable, or pair the highest-R&sup2; pick with business judgment &mdash; sometimes a slightly lower-R&sup2; variable such as `credit_score` is preferred for interpretability, regulatory acceptance, or availability at quote time.